In [15]:
import random
import json
import time
import os
import uuid

from datetime import datetime, timezone
from google import genai
from getpass import getpass

In [33]:
# configuring the gemini client

if not os.environ.get('GEMINI_API_KEY'):
    os.environ['GEMINI_API_KEY']= getpass.getpass('Enter Gemini API key: ')

client = genai.Client()
print('Gemini client initialized successfully!')

Enter Gemini API key:  ········


Gemini client initialized successfully!


In [34]:
MODEL_NAME = "gemini-3.5-flash-lite"
print('Selected model: ', MODEL_NAME)

Selected model:  gemini-3.5-flash-lite


In [4]:
#defining the game configuration

PUZZLE_CATEGORIES = [
    'number_pattern',
    'riddle',
    'logic'
]

DIFFICULTY_LEVELS = [
    'easy',
    'mediium',
    'hard'
]

STARTING_DIFFICULTY = 'easy'

MAX_HINTS = 3
RECENT_HISTORY_LIMIT = 10

print('Puzzle categories:', PUZZLE_CATEGORIES)
print('Difficulty levels:', DIFFICULTY_LEVELS)
print('Starting difficulty:', STARTING_DIFFICULTY)

Puzzle categories: ['number_pattern', 'riddle', 'logic']
Difficulty levels: ['easy', 'mediium', 'hard']
Starting difficulty: easy


In [5]:
#defining game session

def create_game_session():
    session = {
        'session_id': f"session_{int(time.time())}",
        
        'current_difficulty': STARTING_DIFFICULTY,
        
        'total_attempts': 0,
        'correct_answers': 0,
        'incorrect_answers': 0,

        'total_time': 0,
        'total_hints_used':0,

        'attempt_history': [],
        'puzzle_history': [],
        'adaptation_history': []
    }

    return session

game_session = create_game_session()
print(game_session)

{'session_id': 'session_1794943061', 'current_difficulty': 'easy', 'total_attempts': 0, 'correct_answers': 0, 'incorrect_answers': 0, 'total_time': 0, 'total_hints_used': 0, 'attempt_history': [], 'puzzle_history': [], 'adaptation_history': []}


In [6]:
#defining a random category selector

def select_random_category(session):
    previous_category = None

    if session['puzzle_history']:
        previous_category = session['puzzle_history'][-1]['category']

    available_categories = [
        category
        for category in PUZZLE_CATEGORIES
        if category != previous_category
    ]

    selected_category = random.choice(available_categories)

    return selected_category

In [7]:
#defining a common puzzle structure for all categories

def create_puzzle(
    category,
    difficulty,
    question,
    content,
    answer_type,
    correct_answer,
    hints,
    explanation,
    options=None,
    accepted_answers=None,
    source='ai'
):
    puzzle={
        'puzzle_id':str(uuid.uuid4()),

        #Common puzzle information
        'category': category,
        'difficulty': difficulty,
        'question': question,

        #Category-specific information
        'content': content,

        #Answer information
        'answer_type': answer_type,
        'options': options if options is not None else [],
        'correct_answer': correct_answer,
        'accepted_answers': accepted_answers if accepted_answers is not None else [],

        #Hints and explanation
        'hints': hints,
        'explanation': explanation,

        #Puzzle metadata
        'source': source,
        'validation_status': 'pending'
    }

    return puzzle


In [20]:
required_fields = [
    "puzzle_id",
    "category",
    "difficulty",
    "question",
    "content",
    "answer_type",
    "options",
    "correct_answer",
    "accepted_answers",
    "hints",
    "explanation",
    "source",
    "validation_status"
]

for field in required_fields:
    assert field in number_puzzle, f"Missing field: {field}"

assert len(number_puzzle["hints"]) == 3, "A puzzle must have 3 hints."

print("Puzzle structure created successfully!")

Puzzle structure created successfully!


In [21]:
#common puzzle validation

def validate_common_fields(puzzle):
    errors = []

    required_fields = [
        "puzzle_id",
        "category",
        "difficulty",
        "question",
        "content",
        "answer_type",
        "options",
        "correct_answer",
        "accepted_answers",
        "hints",
        "explanation",
        "source",
        "validation_status"
    ]

    #Check required fields
    for field in required_fields:
        if field not in puzzle:
            errors.append(f"Missing required field: {field}")

    #Stop if required fields are missing
    if errors:
        return errors

    #Check category
    if puzzle['category'] not in PUZZLE_CATEGORIES:
        errors.append('Invalid puzzle category')

    #Check difficulty
    if puzzle['difficulty'] not in DIFFICULTY_LEVELS:
        errors.append('Invalid difficulty level')

    #Check question
    if not isinstance(puzzle['question'], str) or not puzzle['question'].strip():
        errors.append('Question must be a non-empty string')

    #Check hints
    if not isinstance(puzzle['hints'], list) or len(puzzle['hints'])!=3:
        errors.append('Each puzzle must contain exactly 3 hints')
    elif not all(isinstance(hint, str) and hint.strip() for hint in puzzle['hints']):
        errors.append('Every hint must be a non-empty string')

    #Check options
    if not isinstance(puzzle['options'], list):
        errors.append('Options must be a list')

    #Check explanation
    if not isinstance(puzzle['explanation'], str) or not puzzle['explanation'].strip():
        errors.append('Explanation must be a non-empty string')

    return errors

    

In [55]:
#number pattern validation

def validate_number_pattern(puzzle):
    errors = []

    content = puzzle['content']

    required_content_fields=[
        'sequence',
        'missing_position',
        'pattern_type'
    ]

    for field in required_content_fields:
        if field not in content:
            errors.append(f"Number pattern is missing: {field}")

    if errors:
        return errors

    sequence = content['sequence']
    missing_position = content['missing_position']

    if not isinstance(sequence,list) or len(sequence)<3:
        errors.append('Sequence must contain at least 3 positions')

    elif not isinstance(missing_position, int) or not (0 <= missing_position < len(sequence)):
        errors.append('Missing position must be a valid zero-based index')

    else:
        #The missing position must be the only None value
        if sequence.count(None) != 1:
            errors.append('Sequence must contain exactly one missing value')

        elif sequence[missing_position] is not None:
            errors.append('The specified missing position must contain None')

    if puzzle['answer_type'] != 'integer':
        errors.append('Number pattern puzzles must have an integer answer type')

    if not isinstance(puzzle['correct_answer'], int) or isinstance(puzzle['correct_answer'], bool):
        errors.append('Correct answer must be an integer')

    if puzzle['options'] and puzzle['correct_answer'] not in puzzle['options']:
        errors.append('Correct answer must be included in the options')

    return errors

In [22]:
#riddles validation

def validate_riddle(puzzle):
    errors = []

    content = puzzle['content']

    if 'riddle_text' not in content:
        errors.append('Riddle is missing riddle_text')
    elif not isinstance(content['riddle_text'], str) or not content['riddle_text'].strip():
        errors.append('Riddle text must be a non-empty string')

    if puzzle['answer_type'] != 'text':
        errors.append('Riddle answer type must be text')

    if not isinstance(puzzle['correct_answer'], str) or not puzzle['correct_answer'].strip():
        errors.append('Riddle correct answer must be non-empty text')

    if not isinstance(puzzle['accepted_answers'], list) or not puzzle['accepted_answers']:
        errors.append('Riddle must have at least one accepted answer')

    return errors

In [23]:
#logic puzzle validation

def validate_logic(puzzle):
    errors=[]

    content = puzzle['content']

    required_content_fields = ['scenario', 'clues']

    for field in required_content_fields:
        if field not in content:
            errors.append(f"Logic puzzle is missing: {field}")

    if errors:
        return errors

    if not isinstance(content['scenario'], str) or not content['scenario'].strip():
        errors.append('Logic scenario must be non-empty text')

    clues = content['clues']

    if not isinstance(clues, list) or not clues:
        errors.append('Logic puzzle must contain at least one clue')

    elif not all (isinstance(clue, str) and clue.strip() for clue in clues):
        errors.append('Every logic clue must be non-empty text')

    if puzzle['answer_type'] not in ['text', 'structured']:
        errors.append('Logic answer type must be text or structured')

    return errors

In [24]:
#final validation function
def validate_puzzle(puzzle):
    errors = validate_common_fields(puzzle)

    if errors:
        return{
            'is_valid': False,
            'errors': errors
        }

    category = puzzle['category']

    if category == 'number_pattern':
        errors.extend(validate_number_pattern(puzzle))

    elif category == 'riddle':
        errors.extend(validate_riddle(puzzle))

    elif category == 'logic':
        errors.extend(validate_logic(puzzle))

    return{
        'is_valid': len(errors) ==0,
        'errors': errors
    }
    

In [2]:
#checking answers

def normalize_text(answer):
    #Standardize text answers before comparison
    if not isinstance(answer, str):
        return ""

    return " ".join(answer.strip().lower().split())

def check_answer(puzzle, player_answer):
    answer_type = puzzle['answer_type']
    correct_answer = puzzle['correct_answer']

    is_correct = False

    #Integer answers, used for number patterns
    if answer_type == 'integer':
        try:
            #Reject values such as 10.5 or ten
            if isinstance(player_answer, bool):
                raise ValueError

            submitted_answer = int(player_answer)

            if str(player_answer).strip() not in [str(submitted_answer), f"+{submitted_answer}"]:
                raise ValueError

            is_correct = submitted_answer == correct_answer

        except(ValueError, TypeError):
            is_correct = False

    #Text answers, used for riddles and some logic puzzles
    elif answer_type == 'text':
        normalized_answer = normalize_text(player_answer)

        accepted_answers = [
            correct_answer
        ] + puzzle.get('accepted_answers', [])

        normalized_accepted = [
            normalize_text(answer)
            for answer in accepted_answers
            if isinstance(answer, str)
        ]

        is_correct = normalized_answer in normalized_accepted

    #Structured answers, used for more complex logic puzzles
    elif answer_type == 'structured':
        is_correct = player_answer == correct_answer

    else: 
        raise ValueError(f"Unsupported answer type: {answer_type}")

    return {
        "is_correct": is_correct,
        "player_answer": player_answer,
        "feedback":(
            "Correct! Well done!"
            if is_correct
            else "Not quite. Try again!"
        ),
        "explanation": puzzle['explanation'] if is_correct else None
    }

In [13]:
number_puzzle = create_puzzle(
    category="number_pattern",
    difficulty="easy",

    question="Find the missing number in the sequence.",

    content={
        "sequence": [2, 4, 6, 8, None],
        "missing_position": 4,
        "pattern_type": "addition"
    },

    answer_type="integer",
    correct_answer=10,

    hints=[
        "Look carefully at how the numbers change.",
        "Compare each number with the number before it.",
        "Each number increases by the same amount."
    ],

    explanation="Each number increases by 2, so the missing number is 10.",

    options=[9, 10, 12, 14],

    source="curated"
)


riddle_puzzle = create_puzzle(
    category="riddle",
    difficulty="easy",
    question="What has cities but no houses, forests but no trees, and rivers but no water?",
    content={
        "riddle_text": "What has cities but no houses, forests but no trees, and rivers but no water?"
    },
    answer_type="text",
    correct_answer="map",
    accepted_answers=["a map"],
    hints=[
        "You might use it when travelling.",
        "It helps you find places.",
        "It shows roads, cities, and rivers."
    ],
    explanation="A map shows cities, forests, and rivers without containing the actual things.",
    source="curated"
)

logic_puzzle = create_puzzle(
    category="logic",
    difficulty="easy",
    question="Which person owns the dog?",
    content={
        "scenario": "Three friends own different pets.",
        "clues": [
            "Anna does not own a cat.",
            "Ben owns a dog.",
            "Cara does not own a dog."
        ]
    },
    answer_type="structured",
    correct_answer={"pet_owner": "Ben"},
    hints=[
        "Look at the clues carefully.",
        "One clue directly identifies the dog owner.",
        "Check what the second clue says about Ben."
    ],
    explanation="The second clue tells us that Ben owns the dog.",
    source="curated"
)



In [14]:
{
    "puzzle_id": "puzzle_123",
    "category": "number_pattern",
    "difficulty": "easy",
    "player_answer": 10,
    "is_correct": True,
    "time_taken": 25,
    "attempts": 1,
    "hints_used": 0,
    "timestamp": "..."
}

{
    "is_correct": true,
    "player_answer": {
        "pet_owner": "Ben"
    },
    "feedback": "Correct! Well done!",
    "explanation": "The second clue tells us that Ben owns the dog."
}


In [26]:
#record attempts

#attempt_history: stores detailed information about every puzzle attempt
#session statistics: provide a quick overview without having to recalculate everything
def record_attempt(
    session,
    puzzle,
    player_answer,
    is_correct,
    time_taken,
    attempts,
    hints_used
):
    attempt_record = {
        "attempt_id": str(uuid.uuid4()),
        "puzzle_id": puzzle['puzzle_id'],
        "category": puzzle['category'],
        "difficulty": puzzle['difficulty'],
        "player_answer": player_answer,
        "is_correct": is_correct,
        "time_taken": time_taken,
        "attempts": attempts,
        "hints_used": hints_used,
        "timestamp": datetime.now(timezone.utc).isoformat()
    }

    #Store individual attempt
    session['attempt_history'].append(attempt_record)

    #Update overall session statistics
    session['total_attempts'] += 1
    session['total_time'] += time_taken
    session['total_hints_used'] += hints_used

    if is_correct:
        session['correct_answers'] += 1
    else:
        session['incorrect_answers'] += 1

    return attempt_record

In [18]:
#a function to summarize the performance
def get_player_performance(session):
    total = session['total_attempts']
    correct = session['correct_answers']

    accuracy = (
        (correct/total)*100
        if total > 0 else 0
    )

    average_time = (
        session['total_time']/total
        if total > 0 else 0
    )

    average_hints = (
        session['total_hints_used']/total
        if total > 0 else 0
    )

    return {
        "total_attempts": total,
        "correct_answers": correct,
        "incorrect_answers": session['incorrect_answers'],
        "accuracy_percentage": round(accuracy, 2),
        "average_time_seconds": round(average_time, 2),
        "average_hints_used": round(average_hints, 2)
    }

In [19]:
#track recent performance
def get_recent_performance(session, limit=5):
    recent_attempts = session['attempt_history'][-limit:]

    if not recent_attempts:
        return{
            "attempts": [],
            "accuracy_percentage": 0
        }

    correct_count = sum(
        1 for attempt in recent_attempts
        if attempt['is_correct']
    )

    accuracy = (
        correct_count / len(recent_attempts)
    ) *100

    return {
        "attempts": recent_attempts,
        "accuracy_percentage": round(accuracy,2)
    }

In [27]:
test_session = create_game_session()

# Simulate three puzzle attempts
record_attempt(
    session=test_session,
    puzzle=number_puzzle,
    player_answer=10,
    is_correct=True,
    time_taken=25,
    attempts=1,
    hints_used=0
)

record_attempt(
    session=test_session,
    puzzle=number_puzzle,
    player_answer=12,
    is_correct=False,
    time_taken=40,
    attempts=1,
    hints_used=1
)

record_attempt(
    session=test_session,
    puzzle=number_puzzle,
    player_answer=10,
    is_correct=True,
    time_taken=30,
    attempts=2,
    hints_used=2
)

{'attempt_id': '91ccc066-3ad4-4f2f-88ce-3f08e5216d54',
 'puzzle_id': '1d0c03df-96ed-4f1c-8659-30e694ea0eac',
 'category': 'number_pattern',
 'difficulty': 'easy',
 'player_answer': 10,
 'is_correct': True,
 'time_taken': 30,
 'attempts': 2,
 'hints_used': 2,
 'timestamp': '2026-11-17T19:31:28.568478+00:00'}

In [28]:
performance = get_player_performance(test_session)

print(json.dumps(performance, indent=4))

{
    "total_attempts": 3,
    "correct_answers": 2,
    "incorrect_answers": 1,
    "accuracy_percentage": 66.67,
    "average_time_seconds": 31.67,
    "average_hints_used": 1.0
}


In [29]:
recent = get_recent_performance(test_session)

print(json.dumps(recent, indent=4))

{
    "attempts": [
        {
            "attempt_id": "f09bea15-98d0-4379-bd9d-627f78182610",
            "puzzle_id": "1d0c03df-96ed-4f1c-8659-30e694ea0eac",
            "category": "number_pattern",
            "difficulty": "easy",
            "player_answer": 10,
            "is_correct": true,
            "time_taken": 25,
            "attempts": 1,
            "hints_used": 0,
            "timestamp": "2026-11-17T19:31:28.568478+00:00"
        },
        {
            "attempt_id": "6cec5031-7750-4faa-9cdf-0867771f2f13",
            "puzzle_id": "1d0c03df-96ed-4f1c-8659-30e694ea0eac",
            "category": "number_pattern",
            "difficulty": "easy",
            "player_answer": 12,
            "is_correct": false,
            "time_taken": 40,
            "attempts": 1,
            "hints_used": 1,
            "timestamp": "2026-11-17T19:31:28.568478+00:00"
        },
        {
            "attempt_id": "91ccc066-3ad4-4f2f-88ce-3f08e5216d54",
            "puzzle_id": 

In [30]:
#completed puzzles: the puzzles that the pplayer actually completes even after multiple attempts
def record_completed_puzzle(
    session,
    puzzle,
    final_answer,
    is_correct,
    time_taken,
    attempts,
    hints_used
):
    puzzle_record = {
        "puzzle_id": puzzle['puzzle_id'],
        "category": puzzle['category'],
        "difficulty": puzzle['difficulty'],
        "final_answer": final_answer,
        "is_correct": is_correct,
        "time_taken": time_taken,
        "attempts": attempts,
        "hints_used": hints_used,
        "completed_at": datetime.now(timezone.utc).isoformat()
    }

    session["puzzle_history"].append(puzzle_record)
    return puzzle_record

In [31]:
def build_adaptation_context(session, recent_limit=5):
    history = session['puzzle_history']
    recent_puzzles = history[-recent_limit:]
    total_completed = len(history)

    correct_count = sum(
        1 for puzzle in history
        if puzzle['is_correct']
    )

    accuracy = (
        correct_count / total_completed * 100
        if total_completed > 0
        else 0
    )

    recent_correct = sum(
        1 for puzzle in recent_puzzles
        if puzzle['is_correct']
    )

    recent_accuracy = (
        recent_correct/len(recent_puzzles) * 100
        if recent_puzzles
        else 0
    )

    return{
        "current_difficulty": session['current_difficulty'],
        "total_puzzles_completed": total_completed,
        "overall_accuracy_percentage": round(accuracy, 2),
        "recent_accuracy_percentage": round(recent_accuracy, 2),
        "recent_puzzles": recent_puzzles
    }

In [32]:
test_session = create_game_session()

record_completed_puzzle(
    session=test_session,
    puzzle=number_puzzle,
    final_answer=10,
    is_correct=True,
    time_taken=25,
    attempts=1,
    hints_used=0
)

record_completed_puzzle(
    session=test_session,
    puzzle=riddle_puzzle,
    final_answer="globe",
    is_correct=False,
    time_taken=55,
    attempts=2,
    hints_used=2
)

record_completed_puzzle(
    session=test_session,
    puzzle=logic_puzzle,
    final_answer={"pet_owner": "Ben"},
    is_correct=True,
    time_taken=35,
    attempts=1,
    hints_used=1
)

adaptation_context = build_adaptation_context(test_session)

print(json.dumps(adaptation_context, indent=4))

{
    "current_difficulty": "easy",
    "total_puzzles_completed": 3,
    "overall_accuracy_percentage": 66.67,
    "recent_accuracy_percentage": 66.67,
    "recent_puzzles": [
        {
            "puzzle_id": "1d0c03df-96ed-4f1c-8659-30e694ea0eac",
            "category": "number_pattern",
            "difficulty": "easy",
            "final_answer": 10,
            "is_correct": true,
            "time_taken": 25,
            "attempts": 1,
            "hints_used": 0,
            "completed_at": "2026-11-17T19:38:57.596358+00:00"
        },
        {
            "puzzle_id": "04e1621f-6f3e-45ba-b461-dc1d3198571e",
            "category": "riddle",
            "difficulty": "easy",
            "final_answer": "globe",
            "is_correct": false,
            "time_taken": 55,
            "attempts": 2,
            "hints_used": 2,
            "completed_at": "2026-11-17T19:38:57.596358+00:00"
        },
        {
            "puzzle_id": "01805ea6-985b-4200-a283-ca968e770a4a",


In [43]:
#adaptation prompt function
def get_adaptation_prompt(context):
    prompt = f"""
    You are the adaptive difficulty manager for Puzzle Pirates, an AI-ppowered brain puzzle game.
    Your job is to analyze player performance and recommend the difficulty of the next puzzle.

    Available difficulty levels:
    - easy
    - medium
    - hard

    Player performance data:
    {json.dumps(context, indent=2)}

    Decision guidelines:

    1. Consider both overall and recent performance.
    2. Consider correctness, completion time, attempts, and hints used.
    3. Consistently strong performance may justify increasing difficulty.
    4. Repeated mistakes or substantial use of hints may justify decreasing difficulty.
    5. Mixed performance may justify maintaining current difficulty.
    6. Do not make a large jump without sufficient evidence.
    7. If the player has completed very few puzzles, be conservative.

    Return ONLY valud JSON in this exact format:

    {{
        "recommended_difficulty":"easy",
        "decision":"maintain",
        "reasoning": "Brief explanation of your decision"
    }}

    The decision must be one of:
    - increase
    - decrease
    - maintain

    The recommended difficulty must be easy, medium, or hard.
    """

    return prompt

In [39]:
def get_ai_difficulty_recommendation(session):
    context = build_adaptation_context(session)

    prompt = get_adaptation_prompt(context)

    response = client.models.generate_content(
        model = MODEL_NAME,
        contents = prompt,
        config={
            "response_mime_type": "application/json"
        }
    )

    recommendation = json.loads(response.text)

    #Validate the AI response
    recommended_difficulty = recommendation.get("recommended_difficulty")

    if recommended_difficulty not in DIFFICULTY_LEVELS:
        raise ValueError("Gemini returned an invalid difficulty")

    if recommendation.get("decision") not in ["increase", "decrease", "maintain"]:
        raise ValueWrror("Gemini returned an invalid decision")

    if not recommendation.get('reasoning'):
        raise ValueError("Gemini did not provide reasoning")

    return recommendation
    

In [46]:
#update the game session
def update_session_difficulty(session, recommendation):
    previous_difficulty = session['current_difficulty']
    new_difficulty = recommendation['recommended_difficulty']

    session['current_difficulty'] = new_difficulty

    adaptation_record = {
        "previous_difficulty": previous_difficulty,
        "new_difficulty": new_difficulty,
        "decision": recommendation['decision'],
        "reasoning": recommendation['reasoning'],
        "timestamp": datetime.now(timezone.utc).isoformat()
    }

    session['adaptation_history'].append(adaptation_record)

    return adaptation_record

In [47]:
recommendation = get_ai_difficulty_recommendation(test_session)

print("AI recommendation:")
print(json.dumps(recommendation, indent=4))

adaptation_result = update_session_difficulty(
    test_session,
    recommendation
)

print("\nUpdated session difficulty:")
print(test_session["current_difficulty"])

print("\nAdaptation record:")
print(json.dumps(adaptation_result, indent=4))

AI recommendation:
{
    "recommended_difficulty": "easy",
    "decision": "maintain",
    "reasoning": "The player has only completed 3 puzzles on easy difficulty with a 66.67% accuracy rate, while using multiple hints and attempts on one of the incorrect puzzles. Given the very few puzzles completed, a conservative approach is best, and mixed performance warrants maintaining the current difficulty level."
}

Updated session difficulty:
easy

Adaptation record:
{
    "previous_difficulty": "easy",
    "new_difficulty": "easy",
    "decision": "maintain",
    "reasoning": "The player has only completed 3 puzzles on easy difficulty with a 66.67% accuracy rate, while using multiple hints and attempts on one of the incorrect puzzles. Given the very few puzzles completed, a conservative approach is best, and mixed performance warrants maintaining the current difficulty level.",
    "timestamp": "2026-11-17T19:56:55.753930+00:00"
}


In [49]:
#number puzzle generation prompt

def get_number_pattern_prompt(difficulty):
    prompt = f"""
    Your are a puzzle creator for Puzzle Pirates, an adaptive brain puzzle game.

    Generate ONE original number pattern puzzle.

    Difficulty: {difficulty}

    Requirements:

    1. The puzzle must have exactly one missing number.
    2. The sequence must contain at least 4 numbers.
    3. The missing number must be represented as null.
    4. Use a clear and mathematically consistent pattern.
    5. The correct answer must be an integer.
    6. Provide exactly 4 multiple-choice options.
    7. Only one option can be correct.
    8. Provide exactly 3 progressive hints.
    9. Hints must gradually reveal the solving approach without immediately giving away the answer.
    10. Provide a clear explanation of the solution.
    11. Avoid ambiguous sequences with multiple valid answers.
    12. Match the complexity of the requested difficulty.

    Return ONLY valid JSON using this structure:
    {{
        "category": "number_pattern",
        "difficulty": "{difficulty}",
        "question": "Find the missing number.",
        "content": {{
            "sequence": [3, 6, 9, null, 15],
            "missing_position": 3,
            "pattern_type": "addition"
        }},
        "answer_type": "integer",
        "correct_answer": 12,
        "options": [10, 11, 12, 14],
        "accepted_answers": [],
        "hints": [
            "Hint 1",
            "Hint 2",
            "Hint 3"
        ],
        "explanation": "Explain the pattern and answer."
    }}

    The example above is only a format reference. Generate a new puzzle rather than copying it.
    """

    return prompt

In [50]:
def generate_number_pattern(difficulty):
    prompt = get_number_pattern_prompt(difficulty)

    response = client.models.generate_content(
        model = MODEL_NAME,
        contents = prompt,
        config= {"response_mime_type": "application/json"}
    )

    if not response.text:
        raise ValueError("Gemini retunred an empty response")

    generated_data = json.loads(response.text)

    return generated_data

In [51]:
generated_data = generate_number_pattern("easy")

print(json.dumps(generated_data, indent=4))

{
    "category": "number_pattern",
    "difficulty": "easy",
    "question": "Find the missing number in the sequence.",
    "content": {
        "sequence": [
            5,
            10,
            15,
            null,
            25
        ],
        "missing_position": 3,
        "pattern_type": "addition"
    },
    "answer_type": "integer",
    "correct_answer": 20,
    "options": [
        18,
        19,
        20,
        22
    ],
    "accepted_answers": [],
    "hints": [
        "Look at how each number changes as you move from left to right.",
        "Compare the difference between the first and second numbers.",
        "Add the same constant value to the number before the missing space."
    ],
    "explanation": "The sequence increases by 5 with each step (5 + 5 = 10, 10 + 5 = 15, 15 + 5 = 20, 20 + 5 = 25). Therefore, the missing number is 20."
}


In [52]:
#convert the AI response to our common puzzle structure using the create_puzzle function

def convert_to_puzzle(generated_data):

    puzzle = create_puzzle(
        category = generated_data['category'],
        difficulty = generated_data['difficulty'],
        question = generated_data['question'],
        content = generated_data['content'],
        answer_type = generated_data['answer_type'],
        correct_answer = generated_data['correct_answer'],
        hints = generated_data['hints'],
        explanation = generated_data['explanation'],
        options = generated_data['options'],
        accepted_answers = generated_data['accepted_answers'],
        source = 'ai'
    )

    return puzzle

In [53]:
number_puzzle_ai = convert_to_puzzle(generated_data)

print(json.dumps(number_puzzle_ai, indent=4))

{
    "puzzle_id": "a2a1e87d-6254-462b-8f95-494bc34a32df",
    "category": "number_pattern",
    "difficulty": "easy",
    "question": "Find the missing number in the sequence.",
    "content": {
        "sequence": [
            5,
            10,
            15,
            null,
            25
        ],
        "missing_position": 3,
        "pattern_type": "addition"
    },
    "answer_type": "integer",
    "options": [
        18,
        19,
        20,
        22
    ],
    "correct_answer": 20,
    "accepted_answers": [],
    "hints": [
        "Look at how each number changes as you move from left to right.",
        "Compare the difference between the first and second numbers.",
        "Add the same constant value to the number before the missing space."
    ],
    "explanation": "The sequence increases by 5 with each step (5 + 5 = 10, 10 + 5 = 15, 15 + 5 = 20, 20 + 5 = 25). Therefore, the missing number is 20.",
    "source": "ai",
    "validation_status": "pending"
}


In [56]:
validation_result = validate_puzzle(number_puzzle_ai)

print(json.dumps(validation_result, indent=4))

{
    "is_valid": true,
    "errors": []
}


In [57]:
if validation_result["is_valid"]:
    number_puzzle_ai["validation_status"] = "passed"
    print("Puzzle generated and validated successfully!")

else:
    number_puzzle_ai["validation_status"] = "failed"
    print("Puzzle validation failed:")

    for error in validation_result["errors"]:
        print("-", error)

Puzzle generated and validated successfully!


In [58]:
print("Category:", number_puzzle_ai["category"])
print("Difficulty:", number_puzzle_ai["difficulty"])
print("Question:", number_puzzle_ai["question"])
print("Sequence:", number_puzzle_ai["content"]["sequence"])
print("Correct answer:", number_puzzle_ai["correct_answer"])

print("\nHints:")
for i, hint in enumerate(number_puzzle_ai["hints"], start=1):
    print(f"{i}. {hint}")

Category: number_pattern
Difficulty: easy
Question: Find the missing number in the sequence.
Sequence: [5, 10, 15, None, 25]
Correct answer: 20

Hints:
1. Look at how each number changes as you move from left to right.
2. Compare the difference between the first and second numbers.
3. Add the same constant value to the number before the missing space.
